# Lecture 8: SVD, Conditioning and Sparsity
Run from top to bottom with NumPy, SciPy and Matplotlib. No external files are required. Figures saved in this notebook are ordinary embedded images, visible in Jupyter and PyCharm. The same executed figures appear on the slides.

We follow one question: what does a matrix preserve, amplify or nearly erase? Conditioning measures sensitivity; SVD identifies the directions and gains; truncation retains selected directions; sparse storage and iterative solving address large systems. Low rank and sparsity describe different structures.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import eye, diags, vstack
from scipy.sparse.linalg import lsqr

rng = np.random.RandomState(587)  # Fixed seed for repeatable examples.
figures = {}

def rotation(degrees):
    angle = np.deg2rad(degrees)
    return np.array([[np.cos(angle), -np.sin(angle)],
                     [np.sin(angle), np.cos(angle)]])


## 1. Sensitivity: two sources and two sensors
Suppose two sources have unknown strengths x1 and x2. Each sensor records a linear combination of their signals: b1=a11*x1+a12*x2 and b2=a21*x1+a22*x2. The known coefficient aij is the response of sensor i to one unit of source j. We collect the equations as A@x=b.

The **measurement matrix A** maps source strengths to predicted sensor readings. Each row represents one sensor, and each column represents all sensors' responses to one source. The observed readings b may contain measurement error. Forward calculation predicts b from x using A@x. Our task goes in the other direction: recover x from known A and observed b. Throughout this discussion, sensitivity means sensitivity of this recovery to measurement errors.

In the example below, the columns of A are almost identical: both sensors respond almost the same way to either source. Can the readings distinguish their strengths reliably? Keep A fixed and change only the second reading by 0.001. We use solve rather than an explicit inverse.

**Consequence for recovery:** measurement errors can cause large errors in the individual source strengths. We cannot reliably separate them from these uncertain readings, even though the exact system has a unique solution.

In [ ]:
A_sensitive = np.array([[1.0, 1.0], [1.0, 1.001]])
b_original = np.array([2.0, 2.001])
b_changed = np.array([2.0, 2.002])
x_original = np.linalg.solve(A_sensitive, b_original)
x_changed = np.linalg.solve(A_sensitive, b_changed)
relative_data_change = np.linalg.norm(b_changed-b_original)/np.linalg.norm(b_original)
relative_solution_change = np.linalg.norm(x_changed-x_original)/np.linalg.norm(x_original)
print("Original coefficients:", x_original)
print("Changed coefficients:", x_changed)
print("Relative measurement change:", relative_data_change)
print("Relative coefficient change:", relative_solution_change)
print("Condition number:", np.linalg.cond(A_sensitive))


In [ ]:
# Compare the sizes of the measurement and coefficient changes.
measurement_change = b_changed - b_original
coefficient_change = x_changed - x_original
print("Measurement change:", measurement_change)
print("Norm of measurement change:", np.linalg.norm(measurement_change))
print("Coefficient change:", coefficient_change)
print("Norm of coefficient change:", np.linalg.norm(coefficient_change))


### Checkpoint: sensitivity
A coefficient change h means replacing x by x+h. The measurement change is A@(x+h)-A@x = A@h. Compare h1=[1,1] and h2=[1,-1]. Calculate A@h1 and A@h2. Which has the smaller Euclidean norm, and therefore leaves the measurements almost unchanged? Both coefficient changes have the same length.

**Answer:** A@h1=[2,2.001], whereas A@h2=[0,-0.001]. Their Euclidean norms are approximately 2.829 and 0.001, respectively, although h1 and h2 have the same length. Increasing one coefficient while decreasing the other nearly cancels in both measurements. Thus a substantial coefficient change can leave the measurements almost unchanged. Sensitivity is part of the equations.

### Checkpoint: matrix or right-hand side?
Keep A fixed and change b to b+delta_b. The solution changes from x to x+delta_x.

1. Does the direction of delta_b matter, or only its magnitude?
2. Keep delta_b fixed. If we change the original b, does delta_x change?
3. Could changing the original b change the relative solution change norm(delta_x)/norm(x)?

**Answers:**

1. **The direction matters.** Measurement changes of the same magnitude can produce solution changes of different magnitudes.
2. **No.** Subtracting the two systems gives A@delta_x=delta_b. With A and delta_b fixed, this determines the same delta_x, regardless of the original b.
3. **Yes.** Changing b changes the original solution x. Therefore norm(delta_x)/norm(x) can change even when delta_x stays the same, assuming x is nonzero.

## 2. Matrix geometry: why a circle?
For a sensitive matrix, our goal is to identify which coefficient directions the measurements detect well and which they barely detect. A stable solver alone cannot supply missing information; weak directions may call for better measurements or a justified restriction on the solution.

We tested two coefficient changes. Now compare all changes h of unit Euclidean length, and measure the length of A@h. In two dimensions the inputs form a unit circle. Their images trace an ellipse (possibly flattened to a line or point). A short output axis means a unit input change produces a small measurement change. Recovery of the corresponding input component is therefore sensitive to measurement errors. Its shortest and longest radii show the weakest and strongest responses. This is why we study the circle-to-ellipse map. SVD identifies these directions and response magnitudes.

An orthogonal factor rotates or reflects, while a diagonal factor stretches coordinate directions. 

### Why an ellipse?
For A=diag(3,0.6), a unit input satisfies h1²+h2²=1. Its image y=A@h has coordinates y1=3*h1 and y2=0.6*h2. Substitution gives (y1/3)²+(y2/0.6)²=1: an ellipse with semiaxes 3 and 0.6. For a general invertible 2-by-2 A, substitute h=A^{-1}y into h.T@h=1 to obtain y.T@(A@A.T)^{-1}@y=1, the equation of an ellipse whose axes may be tilted. This identity explains the geometry; it is not an instruction to form an inverse in the code. A singular matrix can flatten the image into a line segment or a point.

### Introduce the factors before the picture
In two dimensions, the singular value decomposition writes A=U Sigma V.T. Apply it from right to left: V.T expresses the input in the perpendicular unit directions given by the columns of V; the diagonal matrix Sigma multiplies those two coordinates by nonnegative singular values; U places the result along its perpendicular output directions. The square matrices U and V are orthogonal, so their actions rotate or reflect without changing lengths. Sigma changes lengths. Thus the unit circle remains a circle after V.T, becomes an axis-aligned ellipse after Sigma, and gains its final orientation after U. The next figure illustrates these three steps for a matrix with singular values 3 and 0.6.

In [ ]:
A_geometry = rotation(30.0) @ np.diag([3.0, 0.6]) @ rotation(-20.0)
Ug, sg, Vhg = np.linalg.svd(A_geometry, full_matrices=False)
theta = np.linspace(0.0, 2*np.pi, 250)
circle = np.vstack([np.cos(theta), np.sin(theta)])

# The columns of V are the two perpendicular unit input directions.
Vg = Vhg.T
stages = [circle, Vhg @ circle,
          np.diag(sg) @ Vhg @ circle, A_geometry @ circle]
# Track BOTH input directions through the same transformations.
direction_stages = [Vg, Vhg @ Vg,
                    np.diag(sg) @ Vhg @ Vg, A_geometry @ Vg]
labels = [[r"$v_1$", r"$v_2$"], [r"$e_1$", r"$e_2$"],
          [r"$\sigma_1 e_1$", r"$\sigma_2 e_2$"],
          [r"$\sigma_1 u_1$", r"$\sigma_2 u_2$"]]
# Also follow an arbitrary input that is not a principal direction.
h_demo = np.array([np.cos(-0.8), np.sin(-0.8)])
h_stages = [h_demo, Vhg @ h_demo,
            np.diag(sg) @ Vhg @ h_demo, A_geometry @ h_demo]
h_labels = [r"$h$", r"$V^T h$", r"$\Sigma V^T h$", r"$Ah$"]
titles = ["Input directions", "Apply V transpose", "Scale by singular values", "Apply U"]
fig, axs = plt.subplots(1, 4, figsize=(13, 3.4))
for ax, shape, directions, names, title, h_point, h_name in zip(
        axs, stages, direction_stages, labels, titles, h_stages, h_labels):
    ax.plot(shape[0], shape[1], color="gray")
    for i, color in enumerate(["tab:orange", "tab:green"]):
        endpoint = directions[:, i]
        ax.annotate("", xy=endpoint, xytext=(0, 0),
                    arrowprops={"arrowstyle": "->", "color": color})
        offset = (-4, 7) if endpoint[0] > 1.5 else (4, 7)
        alignment = "right" if endpoint[0] > 1.5 else "left"
        ax.annotate(names[i], xy=endpoint, xytext=offset,
                    textcoords="offset points", ha=alignment,
                    color=color, fontsize=12)
    ax.annotate("", xy=h_point, xytext=(0, 0),
                arrowprops={"arrowstyle": "->", "color": "tab:blue"})
    ax.annotate(h_name, xy=h_point, xytext=(4, -16),
                textcoords="offset points", color="tab:blue", fontsize=12)
    ax.axhline(0, color="gray", linestyle=":")
    ax.axvline(0, color="gray", linestyle=":")
    ax.set(xlim=(-3.6, 3.6), ylim=(-3.6, 3.6), title=title)
    ax.set_xticks([-3, 0, 3])
    ax.set_yticks([-3, 0, 3])
    ax.set_aspect("equal")
figures["svd-stages"] = fig
plt.show()


### A non-diagonal example
Let A=[[2,1],[1,2]] and y=A@h, where h1²+h2²=1. Solving for h gives h1=(2*y1-y2)/3 and h2=(2*y2-y1)/3. Substitution into the unit-circle equation gives 5*y1²-8*y1*y2+5*y2²=9. This is a tilted ellipse. With coordinates q=(y1+y2)/sqrt(2) and r=(y1-y2)/sqrt(2), it becomes q²/9+r²=1: semiaxes 3 and 1 along the two diagonal directions. The code draws the image directly by multiplying the unit-circle points by A.

In [ ]:
A_tilted = np.array([[2.0, 1.0], [1.0, 2.0]])
unit_angles = np.linspace(0.0, 2*np.pi, 301)
unit_points = np.array([np.cos(unit_angles), np.sin(unit_angles)])
ellipse_points = A_tilted @ unit_points

fig, axs = plt.subplots(1, 2, figsize=(8, 3))
for ax, points, title in zip(
        axs, [unit_points, ellipse_points],
        ["Unit inputs h", "Outputs y = A @ h"]):
    ax.plot(points[0], points[1])
    ax.axhline(0, color="gray")
    ax.axvline(0, color="gray")
    ax.set_aspect("equal")
    ax.set_xlim(-3.3, 3.3)
    ax.set_ylim(-3.3, 3.3)
    ax.set_title(title)
axs[0].set(xlabel="h1", ylabel="h2")
axs[1].set(xlabel="y1", ylabel="y2")
figures["non-diagonal-ellipse"] = fig
plt.show()


## 3. Singular directions and the Python contract
For a real m-by-n matrix, with q=min(m,n), the reduced decomposition is A=U diag(s) Vh. NumPy returns V transpose, not V. U has shape (m,q), s has shape (q,), and Vh has shape (q,n). The singular values are sorted from largest to smallest and are nonnegative. A right singular vector is a row of Vh, interpreted as a column; A v_i=s_i u_i. Signs of paired singular vectors can change between implementations without changing their product.

### Why SVD exists for every real matrix
The matrix $A^TA$ is symmetric and positive semidefinite: $z^TA^TAz=\|Az\|_2^2\ge0$. By the spectral theorem it has an orthonormal eigenbasis $v_i$, with eigenvalues $\lambda_i\ge0$. Define $\sigma_i=\sqrt{\lambda_i}$.

For each positive $\sigma_i$, set $u_i=Av_i/\sigma_i$. Then $u_i^Tu_j=v_i^TA^TAv_j/(\sigma_i\sigma_j)$ equals 1 when $i=j$ and 0 otherwise. Thus these output vectors are orthonormal, and $Av_i=\sigma_i u_i$.

If $\sigma_i=0$, then $\|Av_i\|_2^2=0$, so $Av_i=0$. Complete the output vectors to an orthonormal basis where needed. These relations give the full SVD $A=U\Sigma V^T$; retaining the first $q=\min(m,n)$ directions gives the reduced form.

### Singular values versus eigenvalues
Even for square matrices they generally differ. Eigenvectors satisfy $Av=\lambda v$, while singular vectors satisfy $Av_i=\sigma_i u_i$, allowing different input and output directions. The singular values are $\sigma_i=\sqrt{\lambda_i(A^TA)}\ge0$.

For $A=\begin{bmatrix}0&1\\0&0\end{bmatrix}$, the eigenvalues are 0,0 but the singular values are 1,0. For a real symmetric matrix, singular values are the absolute values of its eigenvalues, sorted by magnitude. For a symmetric positive semidefinite matrix they coincide. Singular values are always real and nonnegative; eigenvalues of a real matrix may be negative or complex.

In [ ]:
U, s, Vh = np.linalg.svd(A_geometry, full_matrices=False)
reconstructed = U @ np.diag(s) @ Vh
print("Singular values:", s)
print("Shapes:", U.shape, s.shape, Vh.shape)
print("Reconstruction agrees:", np.allclose(A_geometry, reconstructed))
print("U columns orthonormal:", np.allclose(U.T @ U, np.eye(2)))
print("V columns orthonormal:", np.allclose(Vh @ Vh.T, np.eye(2)))
for i in range(2):
    print("Direction", i, "identity holds:",
          np.allclose(A_geometry @ Vh[i], s[i]*U[:, i]))


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 3.5))
axs[0].plot(circle[0], circle[1])
ellipse = A_geometry @ circle
axs[1].plot(ellipse[0], ellipse[1])
for i, color in enumerate(["tab:blue", "tab:orange"]):
    v = Vh[i]
    image_v = s[i]*U[:, i]
    axs[0].quiver(0, 0, v[0], v[1], angles="xy", scale_units="xy", scale=1, color=color)
    axs[1].quiver(0, 0, image_v[0], image_v[1], angles="xy", scale_units="xy", scale=1, color=color)
    axs[0].text(v[0], v[1], " v"+str(i+1))
    axs[1].text(image_v[0], image_v[1], " s"+str(i+1)+" u"+str(i+1))
for ax in axs:
    ax.set_aspect("equal")
axs[0].set(xlim=(-1.5, 1.5), ylim=(-1.5, 1.5), title="Unit input directions")
axs[1].set(xlim=(-3.3, 3.3), ylim=(-3.3, 3.3), title="Output directions and lengths")
figures["singular-directions"] = fig
plt.show()


In [ ]:
# A rectangular matrix maps two input coordinates to three outputs.
A = np.array([[1.0, 0.0], [0.0, 2.0], [1.0, 0.0]])
Ur, sr, Vhr = np.linalg.svd(A, full_matrices=False)
print("Reduced shapes:", Ur.shape, sr.shape, Vhr.shape)
print("Rectangular reconstruction:", np.allclose(Ur @ np.diag(sr) @ Vhr, A))
print("Eigenvalues of A.T @ A:", np.linalg.eigvalsh(A.T @ A))
print("Squared singular values:", sr**2)


### Absolute measurement-error amplification: our original A
For fixed invertible A, measurement errors satisfy $A\Delta x=\Delta b$. Define $\Delta c_i=v_i^T\Delta x$, the coefficient error along input direction $v_i$. From the SVD and orthonormal columns of U, $u_i^TA=\sigma_i v_i^T$. Multiplying $A\Delta x=\Delta b$ by $u_i^T$ therefore gives $u_i^T\Delta b=u_i^TA\Delta x=\sigma_i v_i^T\Delta x=\sigma_i\Delta c_i$. Divide by $\sigma_i>0$ to obtain $\Delta c_i=(u_i^T\Delta b)/\sigma_i$. This is an absolute component error, not a relative error. Only the measurement error component along $u_i$ is amplified by $1/\sigma_i$.

For the original matrix [[1,1],[1,1.001]], the small singular value is approximately 0.000500, and its input/output directions are approximately (1,-1)/sqrt(2). The original reading error [0,0.001] has weak-direction component about -0.000707. Dividing gives a coefficient error about -1.414 along the weak input direction, explaining almost all of the earlier delta_x=[-1,1]. Paired vector signs may reverse without changing the recovered error.

In [ ]:
# Use the original sensor matrix and its original reading error.
U_recovery, s_recovery, Vh_recovery = np.linalg.svd(
    A_sensitive, full_matrices=False)
reading_error = b_changed - b_original

# Separate the measurement error into output-direction components.
measurement_components = U_recovery.T @ reading_error
coefficient_components = measurement_components / s_recovery
recovery_error = Vh_recovery.T @ coefficient_components
print("Singular values:", s_recovery)
print("Measurement-error components:", measurement_components)
print("Coefficient-error components:", coefficient_components)
print("Recovered coefficient error:", recovery_error)
print("Direct solve gives:", np.linalg.solve(A_sensitive, reading_error))


A transpose times A is useful for the mathematical relationship: its eigenvalues are the squared singular values. We do not form it to compute the SVD or solve a sensitive least-squares problem, because its condition number is squared for full-column-rank A. The SVD applies even when A itself is rectangular, singular or not diagonalizable.

Checkpoint: if s=[3,0.6], what are the maximum and minimum output lengths of A x for unit x, and is A rank one? Answer: 3 and 0.6; both directions survive, so rank is two. A small singular value is not an exactly zero one.

## 4. Conditioning
For an invertible square matrix, kappa_2(A)=s_max/s_min. It bounds relative changes in a solution caused by relative changes in b when A stays fixed. It is a worst-case amplification bound, not the error of every solve. A large value can occur without very large entries. Multiplying all equations by a nonzero constant changes the singular values but not their ratio.

### Worked example: a sensitive line fit
Fit b(t)=c0+c1*t using inputs [-0.001,0,0.001]. The true simulated coefficients are [1,0.2]; the clean readings [0.9998,1,1.0002] become [0.9978,1,1.0022] after small measurement errors. The columns of the design matrix are orthogonal, but their lengths differ greatly: singular values sqrt(3) and 0.001*sqrt(2). The weak input direction is the slope coefficient. Over this narrow input range, even a large slope change produces only a small reading change.

Compare retaining both singular directions with a relative cutoff of 0.01. Here the cutoff discards the slope direction; for a general design matrix it discards a singular-vector combination of coefficients, not necessarily a single named coefficient.

In [ ]:
# Build the line-fitting problem before solving it.
t_fit_demo = np.array([-0.001, 0.0, 0.001])
A_fit_demo = np.column_stack([np.ones(3), t_fit_demo])
c_true_demo = np.array([1.0, 0.2])
b_fit_demo = np.array([0.9978, 1.0, 1.0022])

# Compare the usual numerical cutoff with deliberate truncation.
c_full_demo, _, rank_full_demo, s_fit_demo = np.linalg.lstsq(
    A_fit_demo, b_fit_demo, rcond=None)
c_cut_demo, _, rank_cut_demo, _ = np.linalg.lstsq(
    A_fit_demo, b_fit_demo, rcond=0.01)
print("Singular values:", s_fit_demo)
print("Deliberate cutoff:", 0.01*s_fit_demo[0])

# Inspect fit to the observed readings AND error against simulated truth.
for name, coefficients, rank in [
        ("Keep both", c_full_demo, rank_full_demo),
        ("Drop weak direction", c_cut_demo, rank_cut_demo)]:
    print(name)
    print("  Coefficients:", coefficients, "Numerical rank:", rank)
    print("  Data residual norm:", np.linalg.norm(A_fit_demo @ coefficients-b_fit_demo))
    print("  Coefficient error norm:", np.linalg.norm(coefficients-c_true_demo))


**Interpretation:** ordinary least squares gives [1,2.2], fitting these noisy readings exactly but with coefficient error 2. Truncation gives [1,0], leaving a data residual about 0.00311 but reducing coefficient error to 0.2. We know that error only because the true coefficients were supplied in this simulation. The cutoff also removes a genuine slope, so it is not universally better. Choose such a modelling cutoff using noise information or validation; the default cutoff mainly detects numerical rank.

### Checkpoint: fitting data or recovering coefficients?
Use the line-fitting example, whose true coefficients are [1,0.2]. Keeping both directions gives [1,2.2], data residual 0 and coefficient error 2. Dropping the weak direction gives [1,0], data residual about 0.00311 and coefficient error 0.2.

1. Which estimate fits the observed readings better?
2. Which estimate is closer to the true coefficients [1,0.2]?
3. In a real problem we do not know the true coefficients. Does a smaller data residual guarantee more accurate coefficients?

**Answers:**

1. Keeping both directions: the data residual is zero.
2. Dropping the weak direction: coefficient error is 0.2 rather than 2.
3. No. Fitting measurement errors more closely can make coefficient estimates less accurate. The residual measures fit to the readings, not error against the unknown truth.

The cutoff sacrifices some data fit to reduce noise amplification. This is regularisation; it also removes genuine information, so use noise information or validation when choosing a cutoff.

### Default cutoff and float precision
For a matrix with m rows and n columns, the default relative cutoff is $\texttt{rcond}=\epsilon\max(m,n)$. The absolute singular-value threshold is $\sigma_{\mathrm{cutoff}}=\texttt{rcond}\,\sigma_{\max}$. Singular values below that threshold are treated as zero. For our 3-by-2 example these are $3\epsilon$ and $3\epsilon\sigma_{\max}$, respectively.

The [NumPy lstsq documentation](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html) describes the default as machine precision times max(m,n). [np.finfo](https://numpy.org/doc/stable/reference/generated/numpy.finfo.html) provides epsilon: about 2.22e-16 for float64 and 1.19e-7 for float32. For a 3-by-2 matrix, multiplying by 3 gives 6.66e-16 and 3.58e-7 respectively, a factor of about 5.4e8. Multiply a relative cutoff by the largest singular value to obtain the absolute singular-value threshold.

**Important for this particular function:** NumPy 2.5.4 computes `np.linalg.lstsq` internally in double precision. Its `rcond=None` uses float64 epsilon even for float32 inputs. Thus the float32 epsilon is NOT automatically the default cutoff just because the inputs have that dtype. This is verified against the [NumPy implementation](https://github.com/numpy/numpy/blob/main/numpy/linalg/_linalg.py) and the test below. Pass an explicit cutoff if you want it based on input precision.

For our line fit, the default absolute cutoff is about 1.15e-15. An explicit float32-based cutoff gives about 6.19e-7. Both retain the small singular value 0.001414; the deliberately much larger rcond=0.01 discards it.

In [ ]:
# Compare machine epsilons and precision-based cutoff scales.
eps64_demo = np.finfo(np.float64).eps
eps32_demo = np.finfo(np.float32).eps
print("NumPy version:", np.__version__)
for dtype in [np.float64, np.float32]:
    eps = np.finfo(dtype).eps
    relative_cutoff = eps * max(A_fit_demo.shape)
    print(dtype.__name__, "epsilon:", eps)
    print("  Precision-based relative cutoff:", relative_cutoff)
    print("  Absolute cutoff:", relative_cutoff*s_fit_demo[0])

# The two columns here differ enough to expose the cutoff choice.
A_precision_demo = np.diag(np.array([1.0, 1e-8], dtype=np.float32))
b_precision_demo = np.array([1.0, 1e-8], dtype=np.float32)
_, _, rank_none32_demo, _ = np.linalg.lstsq(
    A_precision_demo, b_precision_demo, rcond=None)
rcond32_demo = eps32_demo * max(A_precision_demo.shape)
_, _, rank_explicit32_demo, _ = np.linalg.lstsq(
    A_precision_demo, b_precision_demo, rcond=rcond32_demo)
print("float32 inputs, rcond=None: rank", rank_none32_demo)
print("Explicit float32-based cutoff: rank", rank_explicit32_demo)


In the checked environment the ranks are 2 with None and 1 with the explicit float32-based cutoff. Returning float32 arrays does not imply that the default cutoff uses float32 epsilon. This precision comparison is separate from choosing a cutoff to account for measurement noise.

### Connect SVD to the earlier least-squares command
Recall $\min_x\|Ax-b\|_2$. With rank $r$, the minimum-norm solution is $x^+=\sum_{i=1}^r (u_i^Tb/\sigma_i)v_i$: project b onto each output direction, divide by its nonzero singular value, then combine the input directions.

NumPy's `x, sums, rank, s = np.linalg.lstsq(A, b, rcond=None)` uses an SVD-based solver. `s` contains the singular values of A; `rank` counts the directions retained by its numerical cutoff; `x` is the least-squares answer with smallest norm if there are multiple minimizers. `sums` contains squared residual sums and may be empty; compute `b - A @ x` directly to inspect the residual. `rcond` sets the cutoff relative to the largest singular value. `None` uses machine precision times the larger matrix dimension. We will compare an explicit SVD solve with this command in the noisy-data example below.

[NumPy lstsq reference](https://numpy.org/doc/stable/reference/generated/numpy.linalg.lstsq.html).

## 4. Conditioning
The ratio of the largest to the smallest singular value measures sensitivity for an invertible square matrix. Compare this with the observed change in the first experiment.

### Define the condition number before the singular-value formula
For an invertible square matrix, define $\kappa_2(A)=\|A\|_2\|A^{-1}\|_2$. The matrix 2-norm is $\|A\|_2=\max_{\|h\|_2=1}\|Ah\|_2$: the largest gain on a unit input. SVD shows that it equals $\sigma_{\max}$. The largest gain of the inverse is $1/\sigma_{\min}$. Thus $\kappa_2(A)=\sigma_{\max}/\sigma_{\min}$. For a tall matrix of full column rank, replace the inverse with the pseudoinverse $A^+$; the same ratio applies.

In [ ]:
condition_singular_values = np.linalg.svd(A_sensitive, compute_uv=False)
condition_number = np.linalg.cond(A_sensitive)
print("Sensitive singular values:", condition_singular_values)
print("kappa:", condition_number)
print("Observed amplification:", relative_solution_change/relative_data_change)
print("Bound holds:", relative_solution_change <= condition_number*relative_data_change)
print("Condition of A.T @ A:", np.linalg.cond(A_sensitive.T @ A_sensitive))
print("kappa squared:", condition_number**2)
print("Condition after multiplying by 1000:", np.linalg.cond(1000*A_sensitive))


### Checkpoint: compare condition numbers
The three plots show images of the unit circle, with equal axis scales. Which matrix has the smallest condition number? Which has the largest? Compare the longest and shortest semiaxes.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10, 3.3))
matrices = [rotation(25), np.diag([3.0, 1.0]), np.diag([3.0, 0.03])]
for ax, matrix in zip(axs, matrices):
    image = matrix @ circle
    ax.plot(image[0], image[1])
    ax.set(xlim=(-3.3, 3.3), ylim=(-3.3, 3.3))
    ax.set_aspect("equal")
figures["condition-ellipses"] = fig
plt.show()


**Answers:** the left matrix has the smallest condition number, 1; its image is a circle. The right matrix has the largest, 100; its ellipse has the largest ratio of longest to shortest semiaxis. The middle value is 3. Condition number compares the two gains, rather than the overall size of the image.

## 5. Solving and truncating
For nonzero singular values, the minimum-norm least-squares solution is a sum of contributions (u_i transpose b)/s_i times v_i. Division by a small s_i magnifies disturbances in that measured direction. In the next experiment we know the truth and deliberately add noise along the weak output direction.

In [ ]:
A_noise = rotation(25) @ np.diag([1.0, 0.001]) @ rotation(-15)
Un, sn, Vhn = np.linalg.svd(A_noise, full_matrices=False)
x_truth = Vhn.T @ np.array([1.0, 0.2])
b_clean = A_noise @ x_truth
b_noisy = b_clean + 0.002*Un[:, 1]

# No singular value is discarded in this full-rank calculation.
x_svd = Vhn.T @ ((Un.T @ b_noisy)/sn)
x_lstsq, sums, numerical_rank, returned_s = np.linalg.lstsq(A_noise, b_noisy, rcond=None)
cutoff = 0.01*sn[0]
keep = sn > cutoff
x_truncated = Vhn[keep].T @ ((Un[:, keep].T @ b_noisy)/sn[keep])
for name, estimate in [("All directions", x_svd), ("Truncated", x_truncated)]:
    print(name, "solution error:", np.linalg.norm(estimate-x_truth),
          "data residual:", np.linalg.norm(A_noise @ estimate-b_noisy))
print("Explicit SVD agrees with lstsq:", np.allclose(x_svd, x_lstsq))
print("Returned numerical rank:", numerical_rank, "singular values:", returned_s)


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
positions2 = np.array([0, 1])
axs[0].bar(positions2-0.18, Un.T @ b_clean, 0.36, label="Clean data")
axs[0].bar(positions2+0.18, Un.T @ b_noisy, 0.36, label="Noisy data")
axs[0].set_ylabel("Measurement components in U directions")
axs[0].set_xticks(positions2, ["Strong", "Weak"])
axs[0].legend()
for shift, estimate, label in [(-0.24, x_truth, "Truth"),
                               (0.0, x_svd, "All directions"),
                               (0.24, x_truncated, "Truncated")]:
    axs[1].bar(positions2+shift, Vhn @ estimate, 0.24, label=label)
axs[1].set_ylabel("Recovered components in V directions")
axs[1].set_xticks(positions2, ["Strong", "Weak"])
axs[1].legend()
fig.subplots_adjust(wspace=0.45)
figures["noise-amplification"] = fig
plt.show()


The all-direction answer fits the noisy data exactly but its weak component is wrong by 2. Truncation discards that direction, giving solution error 0.2 instead. It also removes a real component of the truth and leaves a larger data residual. This is regularisation: accept bias to reduce sensitivity. It is not guaranteed to improve every problem, and the cutoff should be justified by noise, accuracy requirements or validation. Floating-point numerical-rank thresholds and deliberate noise filtering are different choices.

## 6. Rank-k approximation
Each term s_i times outer(u_i,v_i) is a rank-one matrix. Keep the first k terms for A_k. This minimises both the spectral and Frobenius approximation errors among matrices of rank at most k. Spectral error equals the next singular value; squared Frobenius error is the sum of the discarded squared singular values. These are approximation guarantees for A, not guarantees about unseen data.

## 6. Best rank-k approximation
Retain the largest singular values and their directions to approximate a matrix with fewer independent patterns. Compare the approximation error with the discarded singular values.

In [ ]:
# An original synthetic grayscale field, not an external photograph.
y_grid = np.linspace(-1, 1, 96)
x_grid = np.linspace(-1, 1, 128)
X_grid, Y_grid = np.meshgrid(x_grid, y_grid)
field = (np.exp(-4*((X_grid+0.4)**2 + (Y_grid-0.1)**2))
         + 0.8*np.exp(-15*((X_grid-0.35)**2 + (Y_grid+0.3)**2))
         + 0.15*np.sin(4*np.pi*X_grid)*np.cos(2*np.pi*Y_grid))
field = (field-field.min())/(field.max()-field.min())
A_image = field + rng.normal(0.0, 0.035, field.shape)
Ui, si, Vhi = np.linalg.svd(A_image, full_matrices=False)
image_rank = np.linalg.matrix_rank(A_image)

def rank_k(k):
    return Ui[:, :k] @ np.diag(si[:k]) @ Vhi[:k, :]

print("Image matrix shape:", A_image.shape)
print("Original numerical rank:", image_rank)
print("Smallest singular value:", si[-1])
for k in [1, 5, 15]:
    approximation = rank_k(k)
    measured = np.linalg.norm(A_image-approximation, "fro")
    predicted = np.sqrt(np.sum(si[k:]**2))
    print("k:", k, "Frobenius error:", measured, "tail formula:", predicted)


In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(12, 3))
for ax, matrix, title in zip(axs,
        [A_image, rank_k(1), rank_k(5), rank_k(15)],
        ["Original: rank "+str(image_rank), "k = 1", "k = 5", "k = 15"]):
    ax.imshow(matrix, cmap="gray", vmin=A_image.min(), vmax=A_image.max())
    ax.set_title(title)
    ax.axis("off")
figures["rank-images"] = fig
plt.show()


In [ ]:
# Each outer product is a matrix with rank one.
term1 = si[0]*np.outer(Ui[:, 0], Vhi[0])
term2 = si[1]*np.outer(Ui[:, 1], Vhi[1])
matrices_to_show = [term1, term2, term1+term2]
# Use the same colour scale for all three matrices.
limit = max(np.max(np.abs(matrix)) for matrix in matrices_to_show)
fig, axs = plt.subplots(1, 3, figsize=(11, 3.7))
fig.subplots_adjust(left=0.06, right=0.86, bottom=0.18, top=0.78, wspace=0.35)
for ax, matrix, title in zip(
        axs, matrices_to_show,
        [r"First term: $\sigma_1 u_1 v_1^T$",
         r"Second term: $\sigma_2 u_2 v_2^T$",
         r"Their sum: $A_2$ (rank at most 2)"]):
    colour_image = ax.imshow(matrix, cmap="coolwarm", vmin=-limit, vmax=limit)
    ax.set_title(title)
    ax.set_xlabel("Column index")
    ax.set_ylabel("Row index")
    ax.set_xticks([0, matrix.shape[1]-1])
    ax.set_yticks([0, matrix.shape[0]-1])
# One colourbar explains the numerical values in every panel.
colourbar_ax = fig.add_axes([0.9, 0.2, 0.018, 0.54])
fig.colorbar(colour_image, cax=colourbar_ax, label="Matrix entry value")
figures["rank-terms"] = fig
plt.show()


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.4))
axs[0].plot(np.arange(1, 21), si[:20], "o-")
axs[0].set(xlabel="Index i", ylabel="Singular value", title="First 20 of "+str(len(si))+" singular values")
ks = np.arange(0, 21)
tail_errors = np.array([np.sqrt(np.sum(si[k:]**2)) for k in ks])
axs[1].plot(ks, tail_errors, "o-")
axs[1].set(xlabel="Retained rank k", ylabel="Frobenius approximation error")
figures["singular-spectrum"] = fig
plt.show()

k = 5
print("Original scalar values:", A_image.size)
print("Rank-5 factor values:", k*(A_image.shape[0]+A_image.shape[1]+1))
print("Retained squared Frobenius fraction:", np.sum(si[:k]**2)/np.sum(si**2))


Store the factors of A_k=U_k Sigma_k V_k.T: U_k has m*k values, the diagonal of Sigma_k has k values (not k²), and V_k.T has k*n values. Total m*k+k+k*n=k*(m+n+1). For m=96, n=128 and k=5, this is 5*(96+128+1)=1125 values, compared with 96*128=12288 for the full matrix. Keep the factors to obtain the saving; multiplying them out and storing the full matrix again uses m*n values. These counts describe stored numerical values, not file-format overhead.

## 7. Two kinds of structure
An identity matrix is sparse but full rank. A matrix of ones is dense but rank one. Sparse storage and low-rank factors exploit different properties.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(7, 3))
axs[0].imshow(np.eye(12), cmap="gray", vmin=0, vmax=1)
axs[0].set_title("Identity: sparse, rank 12")
axs[1].imshow(np.ones((12, 12)), cmap="gray", vmin=0, vmax=1)
axs[1].set_title("All ones: dense, rank 1")
for ax in axs:
    ax.set_xticks([])
    ax.set_yticks([])
figures["two-structures"] = fig
plt.show()


## 8. The same recovery problem at a larger scale
Earlier examples explained sensitivity and recovery with SVD on moderate-sized dense matrices. Now we still solve min ||A@x-b||, but there are thousands of unknowns. Let x_i be the value at location i along a line, for example a temperature. One set of noisy sensors reads individual locations d_i approximately x_i; another reads neighbouring averages a_i approximately (x_i+x_(i+1))/2. Because of noise, the readings may disagree. Least squares finds one set of values that fits both sets of readings.

For p unknown values, there are p direct readings and p-1 averages: the matrix has shape (2*p-1,p). The direct block has p nonzeros and the average block 2*(p-1), giving 3*p-2 total. Each row uses only one or two unknowns, making the matrix sparse. At p=5000 this is 9999-by-5000: almost 50 million dense entries, but only 14998 nonzeros. The following code stores those nonzeros, then uses LSQR, which needs products with A and A.T rather than a full dense factorisation. Sparse storage reduces memory and makes those products cheap. This is the connection between the measurement model, sparse storage and the iterative solver.

In [ ]:
def sensor_matrix(p):
    direct = eye(p, format="csr")
    averages = diags([0.5*np.ones(p-1), 0.5*np.ones(p-1)],
                     [0, 1], shape=(p-1, p), format="csr")
    return vstack([direct, averages], format="csr")

# First inspect a tiny version; only here do we turn it into a dense array.
print(sensor_matrix(4).toarray())

### Computational cost
LSQR is an iterative linear least-squares algorithm that uses matrix-vector products and small QR steps internally. We do not need its algorithmic derivation here.

For a tall $m\times n$ matrix ($m\ge n$), dense QR factorisation costs $O(mn^2)$. With $k$ iterations, dense LSQR costs about $O(kmn)$. Sparse LSQR costs about $O(k\,\mathrm{nnz}(A))$ for the matrix-vector products, plus vector operations; $\mathrm{nnz}(A)$ counts nonzero entries. Vector work costs $O(k(m+n))$.

When $k\ll n$, iteration can reduce work substantially. The number of iterations depends on accuracy, scaling and conditioning, so a large or sparse matrix does not by itself guarantee that LSQR is faster. These are work estimates rather than measured runtime guarantees.

### See the small blocks and CSR storage
Use three locations to inspect the entire matrix. `eye(3)` creates the identity; `diags` puts 0.5 on offsets 0 and 1 of a two-row, three-column matrix. `vstack` places that block below the identity.

CSR means compressed sparse row. It stores `data` (values), `indices` (column numbers) and `indptr` (row boundaries in the first two arrays). These are zero-based array positions, not memory addresses. In this example, zeros are omitted. The matrix remains mathematically the same. Only use `.toarray()` here because this example is tiny.

In [ ]:
I_demo = eye(3, format="csr")
weights = 0.5*np.ones(2)
B_demo = diags([weights, weights], [0, 1], shape=(2, 3), format="csr")
A_demo = vstack([I_demo, B_demo], format="csr")
print("Identity block:")
print(I_demo.toarray())
print("Adjacent-average block:")
print(B_demo.toarray())
print("Stacked matrix:")
print(A_demo.toarray())
print("Values:", A_demo.data)
print("Column indices:", A_demo.indices)
print("Row pointers:", A_demo.indptr)

### Construct indptr by counting
Start with the list [0]. Read the matrix row by row. After each row, append the total number of nonzeros stored so far. The five rows contain 1,1,1,2,2 nonzeros:

- Before reading any row: 0.
- After row 0: 0+1=1.
- After row 1: 1+1=2.
- After row 2: 2+1=3.
- After row 3: 3+2=5.
- After row 4: 5+2=7.

Collect the initial zero and these totals: indptr=[0,1,2,3,5,7]. Its entry indptr[i] counts stored entries BEFORE matrix row i. So row 3 begins after 3 stored entries and ends after 5: it occupies storage positions 3 and 4, the slice 3:5. The final pointer 7 is the total number of stored entries. SciPy builds this automatically when we construct a CSR matrix.

In [ ]:
# Build the pointers by counting, to understand the small example.
nonzeros_per_row = [1, 1, 1, 2, 2]
pointers_by_count = [0]
for count in nonzeros_per_row:
    pointers_by_count.append(pointers_by_count[-1] + count)
print("Constructed indptr:", pointers_by_count)
print("SciPy indptr:", A_demo.indptr)


In [ ]:
row_demo = 3
start_demo = A_demo.indptr[row_demo]
end_demo = A_demo.indptr[row_demo + 1]
print("Values in row 3:", A_demo.data[start_demo:end_demo])
print("Their columns:", A_demo.indices[start_demo:end_demo])

### Where does the row number come from?
The row number is i, the index into indptr, not the stored value indptr[i]. Values in indptr are positions in data and indices. Begin with a zero matrix. Row 0 uses storage positions 0:1; row 1 uses 1:2; row 2 uses 2:3; row 3 uses 3:5; row 4 uses 5:7. At each position j, indices[j] gives the column and data[j] gives the value to write.

For row 3, positions 3 and 4 have columns 0 and 1 and values 0.5 and 0.5. Thus A[3,0]=0.5 and A[3,1]=0.5, with the remaining entry in row 3 equal to zero. The row stays 3 while we read both storage positions.

In [ ]:
# Reconstruct only the small demonstration matrix, not the large one.
A_rebuilt_demo = np.zeros(A_demo.shape)
for row in range(A_demo.shape[0]):
    # The array index 'row' identifies the matrix row.
    start = A_demo.indptr[row]
    stop = A_demo.indptr[row + 1]
    for position in range(start, stop):
        # These arrays supply the column and value for that row.
        column = A_demo.indices[position]
        value = A_demo.data[position]
        A_rebuilt_demo[row, column] = value
print("Reconstructed matrix:")
print(A_rebuilt_demo)
print("Agrees with the original:", np.allclose(A_rebuilt_demo, A_demo.toarray()))


In [ ]:
p = 5000
positions = np.linspace(0.0, 1.0, p)
true_signal = np.sin(2*np.pi*positions) + 0.3*np.cos(6*np.pi*positions)
A_sparse = sensor_matrix(p)
b_observed = A_sparse @ true_signal + rng.normal(0.0, 0.02, A_sparse.shape[0])
print("Matrix shape:", A_sparse.shape)
print("Stored nonzeros:", A_sparse.nnz)
# These memory counts refer to matrix storage only, not the whole solver.
dense_bytes = A_sparse.shape[0]*A_sparse.shape[1]*8
sparse_bytes = A_sparse.data.nbytes + A_sparse.indices.nbytes + A_sparse.indptr.nbytes
print("Dense float64 storage, MB:", dense_bytes/1e6)
print("CSR array storage, MB:", sparse_bytes/1e6)

### What is the true signal?
The horizontal axis is spatial position along a line, from 0 to 1. The vertical axis is the value we want to recover at that position. The known curve generates 5000 true values x_i. We use A@true_signal to generate direct readings and neighbouring averages, then add noise. LSQR recovers those 5000 values from the readings. We know the truth only because this is a simulation.

In [ ]:
# Show the full signal before trying to recover it from noisy measurements.
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(positions, true_signal)
ax.set(xlabel="Position along the line", ylabel="True value", xlim=(0.0, 1.0))
figures["sparse-true-signal"] = fig
plt.show()


### LSQR inputs and outputs
`lsqr(A, b, atol=1e-6, btol=1e-6, iter_lim=None, show=False)` shows selected arguments. It solves linear least squares using products with `A` and its transpose, without explicitly forming `A.T @ A` or a dense factorisation.

`atol` and `btol` control scaled stopping tests, not direct bounds on the unknown solution error. `iter_lim` limits work; `show=True` displays progress. LSQR returns a tuple, unlike the result objects from `root` and `least_squares`:

- `[0]`: solution; `[1]`: stopping code; `[2]`: iteration count; `[3]`: residual norm.
- Common codes 1 and 2 indicate approximate equation/least-squares solutions. Code 7 means the iteration limit was reached. Other codes are documented; always read the status.

A sparse representation helps only when most entries are zero. An iterative solver need not be faster on a small dense problem.

For this example, the solution has shape `(5000,)`. `np.linalg.norm(A @ x - b)` independently measures the residual reported at tuple index 3. The reconstruction plot shows the full spatial interval, then a separate panel of estimated minus true values to make the small errors visible. LSQR may stop when the equations are approximately satisfied, or when the residual is approximately stationary for least squares; a nonzero residual is expected for incompatible noisy data.

In [ ]:
lsqr_result = lsqr(A_sparse, b_observed, atol=1e-8, btol=1e-8, iter_lim=100)
signal_fit = lsqr_result[0]
print("Stopping code:", lsqr_result[1])
print("Iterations:", lsqr_result[2])
print("Residual norm:", lsqr_result[3])
print("Explicit residual norm:", np.linalg.norm(A_sparse @ signal_fit - b_observed))

In [ ]:
# Compare the full reconstructed signal with the known simulated truth.
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
fig.subplots_adjust(wspace=0.4)
axs[0].plot(positions, true_signal, label="True values")
axs[0].plot(positions, signal_fit, label="LSQR estimate")
axs[0].set(xlabel="Position along the line", ylabel="Value", xlim=(0.0, 1.0),
           title="Full signal: true and recovered")
axs[0].legend(loc="upper right")

# Their difference is easier to see when the two curves nearly overlap.
recovery_error_signal = signal_fit - true_signal
axs[1].plot(positions, recovery_error_signal)
axs[1].axhline(0, color="gray", linestyle="--")
axs[1].set(xlabel="Position along the line", ylabel="Estimate minus true value",
           xlim=(0.0, 1.0), title="Error in recovered values")
figures["sparse-signal"] = fig
plt.show()


In [ ]:
# Verify agreement on a small version that can safely be densified.
small_A = sensor_matrix(40)
small_truth = np.sin(np.linspace(0.0, 2*np.pi, 40))
small_b = small_A @ small_truth + rng.normal(0.0, 0.02, small_A.shape[0])
dense_answer = np.linalg.lstsq(small_A.toarray(), small_b, rcond=None)[0]
iterative_answer = lsqr(small_A, small_b, atol=1e-12, btol=1e-12)[0]
print("Dense and iterative answers agree:",
      np.allclose(dense_answer, iterative_answer, rtol=1e-8, atol=1e-9))
print("Dense residual:", np.linalg.norm(small_A @ dense_answer-small_b))
print("LSQR residual:", np.linalg.norm(small_A @ iterative_answer-small_b))

## Try it
1. Set `iter_lim=1` in LSQR and inspect the stopping code and answer. Restore it before comparing converged solutions.
2. Increase the problem size and compare matrix-storage estimates. Do not densify the large matrix.
3. Relate iteration counts and requested accuracy to the conditioning discussed above.

## What to connect
The dense SVD explains geometry, sensitivity and dominant directions. It does not mean computing a full dense SVD is the right way to handle a huge sparse matrix. For the large sensor system, LSQR uses sparse products with A and A transpose. Its convergence depends on conditioning and the requested accuracy. This particular sensor matrix includes an identity block and is well-conditioned: its singular values lie between 1 and sqrt(2). Sparsity alone would not establish that.

Main references: Stanford EE263 notes on SVD and low-rank approximation, NumPy SVD/cond/lstsq, and SciPy sparse storage/LSQR. The slide reading list gives clickable links. All figures here are generated by the code above.